# Binarization with scBoolSeq

1) Set the GO per macrostate for the evaluation of the HVG and binarization results 
2) Binarize the matrix, the workflow is based on the raw matrix, and the macrostates are binarized separately
3) Evaluate the binarization result 

In [1]:
# === PARAMETERS ===
input_file = "/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Matrix/cll_raw_macro.h5ad"
patient = "P2"
output_directory = "/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Plots"
macrostates_computed="timepoint" # stream2 or timepoint
binarization ="scBoolSeq" # type of binarization choosen : scBoolSeq or classic (based on the count)

In [2]:
import sys
#!pip install gseapy
#!pip install scboolseq
import scanpy as sc
import numpy as np
import pandas as pd
from scboolseq import scBoolSeq
import gc
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
import gseapy as gp
from gseapy import get_library
from scipy.stats import fisher_exact
import json
#! pip install goatools
#!pip install leidenalg

In [3]:
warnings.filterwarnings("ignore")
adata = sc.read(input_file) # read data 
if adata.X.max() > 10 : # check if is raw data, need to be > 10
    # Normalization
    sc.pp.normalize_total(adata) # 
    sc.pp.log1p(adata)
    print("=== This dataset is now normalised")
else :
    print("=== This dataset appears to have already been normalised")

# Verify if we have only the cells expected and the macrostates 
print("=== Cell type and macrostates in our dataset")
print(adata.obs["Annotation"].unique())
print(adata.obs["macrostates"].unique())

=== This dataset is now normalised
=== Cell type and macrostates in our dataset
['B intermediate' 'B naive' 'B' 'B memory' 'Bridge']
['I2' 'None' 'I1' 'T1' 'T3' 'T2']


# Definition of the majority vote

In [18]:
if binarization == "scBoolSeq":
    warnings.filterwarnings("ignore")

    macrostates = ['I1', 'I2', 'T1', 'T2', "T3"]

    binarized_states = {}
    all_hvg = set()
    adata_ct_dict = {}

    for ct in macrostates:
        print(f"\n{'='*50}")
        print(f"Processing {ct}...")

        adata_ct = adata[adata.obs['macrostates'] == ct].copy()
        n_cells = adata_ct.n_obs
        print(f"  {n_cells} cells")

        # STEP 1 : HVG
        n_top = min(2000, adata_ct.n_vars - 1)
        sc.pp.highly_variable_genes(adata_ct, n_top_genes=n_top)

        adata_ct_hvg = adata_ct[:, adata_ct.var['highly_variable']].copy()
        hvg_genes = adata_ct.var[adata_ct.var['highly_variable']].index
        all_hvg.update(hvg_genes)
        print(f"  {adata_ct_hvg.n_vars} HVGs selected")

        # STEP 2 : DataFrame
        X_full = adata_ct_hvg.X
        if not isinstance(X_full, np.ndarray):
            X_full = X_full.toarray()

        expr_df_full = pd.DataFrame(
            X_full,
            index=adata_ct_hvg.obs_names,
            columns=adata_ct_hvg.var_names
        )

        # Remove all-zero genes
        expr_df_full = expr_df_full.loc[:, (expr_df_full != 0).any(axis=0)]
        print(f" {expr_df_full.shape[1]} genes after removing all-zero")

        # STEP 3 : scBoolSeq model
        scbool = scBoolSeq(
            zeroinf_binarizer="quantile",
            margin_quantile=0.2,
            dor_threshold=0.85,
            alpha=0
        )

        scbool.fit(expr_df_full)
        
        print("  scBoolSeq fitted")

        # STEP 4 : Binarization
        binarized = scbool.binarize(expr_df_full)

        #print(binarized.head())

        # STEP 5 : produce plot
        n_cells = len(binarized)
        proportions = pd.DataFrame({
            '0': binarized.apply(lambda col: (col == 0.0).sum() / n_cells).values,
            '1': binarized.apply(lambda col: (col == 1.0).sum() / n_cells).values,
            'NaN': binarized.apply(lambda col: col.isna().sum() / n_cells).values
        })

        # Order
        proportions = proportions.sort_values(['0', '1'], ascending=[False, False]).reset_index(drop=True)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        x = np.arange(len(proportions))

        ax.bar(x, proportions['0'], width=1.0, label='0', color='orange', alpha=0.8)
        ax.bar(x, proportions['1'], width=1.0, bottom=proportions['0'], label='1', color='green', alpha=0.8)
        ax.bar(x, proportions['NaN'], width=1.0, bottom=proportions['0'] + proportions['1'], label='NaN', color='gray', alpha=0.8)

        ax.set_xlim(-0.5, len(proportions) - 0.5)
        ax.margins(x=0)
        
        ax.set_ylabel('Proportion', fontsize=16)
        ax.set_xlabel('Genes', fontsize=16)
        ax.set_title(f'Macrostate {ct}', fontsize=20)
        ax.set_xticks([])
        ax.tick_params(axis='y', labelsize=14) 
        
        # Legend
        ax.legend(title='Value', loc='upper right')
    
        plt.tight_layout()
        plt.savefig(f'{output_directory}/proportions_{ct}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)

        del adata_ct, expr_df_full, scbool
        gc.collect()
        print("\nDone!")


Processing I1...
  5018 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted

Done!

Processing I2...
  1015 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted

Done!

Processing T1...
  264 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted

Done!

Processing T2...
  1797 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted

Done!

Processing T3...
  1699 cells
  2000 HVGs selected
 2000 genes after removing all-zero
  scBoolSeq fitted

Done!


In [7]:
all_genes = adata.var_names
hvg_mask = np.array([gene in all_hvg for gene in all_genes])
print(f"Total genes: {len(all_genes)}")
print(f"HVG macrostates: {hvg_mask.sum()} genes")

Total genes: 18723
HVG macrostates: 6214 genes


Binarization basic count >=1 =1, count=0 =0

In [11]:
if binarization == "classic":

    warnings.filterwarnings("ignore")

    macrostates = ['I1', 'I2', 'T1', 'T2', "T3"]

    binarized_states = {}
    all_hvg = set()
    adata_ct_dict = {}

    for ct in macrostates:
        print(f"\n{'='*50}")
        print(f"Processing {ct}...")
        
        adata_ct = adata[adata.obs['macrostates'] == ct].copy()
        n_cells = adata_ct.n_obs
        print(f"  {n_cells} cells")

        # STEP 1 : HVG adapted to the cell type size
        n_top = min(2000, adata_ct.n_vars - 1)
        sc.pp.highly_variable_genes(adata_ct, n_top_genes=n_top)

        adata_ct_hvg = adata_ct[:, adata_ct.var['highly_variable']].copy()
        hvg_genes = adata_ct.var[adata_ct.var['highly_variable']].index
        all_hvg.update(hvg_genes)
        print(f"  {adata_ct_hvg.n_vars} HVGs selected")

        # STEP 2 : Building the DataFrame
        X_full = adata_ct_hvg.X
        if not isinstance(X_full, np.ndarray):
            X_full = X_full.toarray()

        expr_df_full = pd.DataFrame(
            X_full,
            index=adata_ct_hvg.obs_names,
            columns=adata_ct_hvg.var_names
        )

        # Eliminate the 0 genes
        expr_df_full = expr_df_full.loc[:, (expr_df_full != 0).any(axis=0)]
        print(f" {expr_df_full.shape[1]} genes after removing all-zero")

        # STEP 3 : Binarize per macrostate
        # Binairization with a threshold : ~0 UMI (log(1))
        threshold = np.log(1)
        binarized = (expr_df_full > threshold).astype(int)

        print(f" {binarized.shape[1]} genes binarized")
        print(f"  Binarization done")

        # STEP 4 : Majority vote on all the cells
        def majority_vote(col, threshold=0.3, max_nan=0, min_valid=10):
            n_total = len(col)
            n_nan = col.isna().sum()
            nan_frac = n_nan / n_total
            # Too many Nan = Nan (70%)
            if nan_frac > max_nan:
                return np.nan
            valid = col.dropna()
            # Too few cells (10)
            if len(valid) < min_valid:
                return np.nan
                
            frac_ones = (valid == 1).mean()

            if frac_ones >= threshold:
                return 1
            elif frac_ones <= (1 - threshold):
                return 0
            else:
                return np.nan

        aggregated = binarized.apply(majority_vote, axis=0)
        binarized_states[ct] = aggregated.to_dict()

        n_ones  = (aggregated == 1).sum()
        n_zeros = (aggregated == 0).sum()
        n_nan   = aggregated.isna().sum()
        print(f"  {ct}: {n_cells} cells → {n_ones} genes=1, {n_zeros} genes=0, {n_nan} genes=NaN")

        del adata_ct, expr_df_full
        gc.collect()

    print("\nDone!")


Processing I1...
  5018 cells
  2000 HVGs selected
 2000 genes after removing all-zero
 2000 genes binarized
  Binarization done
  I1: 5018 cells → 428 genes=1, 1572 genes=0, 0 genes=NaN

Processing I2...
  1015 cells
  2000 HVGs selected
 2000 genes after removing all-zero
 2000 genes binarized
  Binarization done
  I2: 1015 cells → 523 genes=1, 1477 genes=0, 0 genes=NaN

Processing T1...
  264 cells
  2000 HVGs selected
 2000 genes after removing all-zero
 2000 genes binarized
  Binarization done
  T1: 264 cells → 204 genes=1, 1796 genes=0, 0 genes=NaN

Processing T2...
  1797 cells
  2000 HVGs selected
 2000 genes after removing all-zero
 2000 genes binarized
  Binarization done
  T2: 1797 cells → 262 genes=1, 1738 genes=0, 0 genes=NaN

Processing T3...
  1699 cells
  2000 HVGs selected
 2000 genes after removing all-zero
 2000 genes binarized
  Binarization done
  T3: 1699 cells → 468 genes=1, 1532 genes=0, 0 genes=NaN

Done!
